<a href="https://colab.research.google.com/github/etmcrae/Who-Got-DOGEd-/blob/main/Linear_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#import libraries
#import all libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn import datasets
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder

In [10]:
#import df_mod2 from google drive
from google.colab import drive
drive.mount('/content/drive')
df_mod3 = pd.read_csv('/content/drive/MyDrive/DOGE/df_mod3.csv')
df_merged = pd.read_csv('/content/drive/MyDrive/DOGE/df_merged.csv')
df_mod2 = pd.read_csv('/content/drive/MyDrive/DOGE/df_mod2.csv')
contracts = pd.read_csv('/content/drive/MyDrive/DOGE/contracts.csv')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipython-input-10-3305672714.py:5: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df_merged = pd.read_csv('/content/drive/MyDrive/DOGE/df_merged.csv')


In [14]:
contracts_2 = contracts.copy()
#converts delete date to a date format
contracts_2['deleted_date'] = pd.to_datetime(contracts_2['deleted_date'])

In [20]:
#adds flag column in contracts_2 called early_deletion
contracts_2['early_deletion'] = 0
#if deleted_date occurs in the months of 01 or 02
contracts_2.loc[(contracts_2['deleted_date'].dt.month == 1) | (contracts_2['deleted_date'].dt.month == 2), 'early_deletion'] = 1

In [22]:
contracts_2.head(100)

,piid,agency,vendor,value,description,fpds_status,fpds_link,deleted_date,savings,early_deletion
0,2032H824F00058,Department of Treasury,DELOITTE CONSULTING LLP,7.259739e+06,Comprehensive Training Strategy aimed at enhan...,TERMINATED,https://www.fpds.gov/ezsearch/jsp/viewLinkCont...,2025-06-03,4.354402e+06,0
1,20341522F00039,Department of Treasury,"FI CONSULTING, INC.",2.992678e+06,Independent Verification and Validation,TERMINATED,https://www.fpds.gov/ezsearch/jsp/viewLinkCont...,2025-06-03,0.000000e+00,0
2,75D30124F19709,Department of Health and Human Services,FAMILY HEALTH INTERNATIONAL,1.428250e+08,HUMAN IMMUNODEFICIENCY VIRUS COMMUNICATIONS PR...,TERMINATED,https://www.fpds.gov/ezsearch/jsp/viewLinkCont...,2025-06-03,1.245576e+08,0
3,Unavailable,USAID,Unavailable,9.499996e+07,Unavailable for legal reasons,Unavailable,https://fpds.gov,2025-06-03,6.524996e+07,0
4,95170024C0273,United States Agency for Global Media,MISCELLANEOUS FOREIGN AWARDEES,1.767699e+07,MEDIUM WAVE RADIO BROADCASTING TRANSMISSION SE...,TERMINATED,https://www.fpds.gov/ezsearch/jsp/viewLinkCont...,2025-06-03,1.608278e+07,0
...,...,...,...,...,...,...,...,...,...,...
95,89303023FIM000115,Department of Energy,ACCENTURE FEDERAL SERVICES LLC,1.299794e+07,Electronic Records Migration Inventory Compliance,FUNDING ONLY ACTION,https://www.fpds.gov/ezsearch/jsp/viewLinkCont...,2025-05-30,2.187142e+06,0
96,47QPCA23F0029,General Services Administration,BIXAL SOLUTIONS INCORPORATED,1.120453e+06,Analytics.usa.gov and Touchpoints,FUNDING ONLY ACTION,https://www.fpds.gov/ezsearch/jsp/viewLinkCont...,2025-05-30,5.586186e+05,0
97,47QPCA24F0011,General Services Administration,BIXAL SOLUTIONS INCORPORATED,8.615795e+06,Vote.gov modernization efforts,CHANGE ORDER,https://www.fpds.gov/ezsearch/jsp/viewLinkCont...,2025-05-30,7.158003e+05,0
98,47QPCA24F0015,General Services Administration,"NOBLIS, INC.",6.315093e+07,FedRAMP Program Management & Technical ISSO su...,FUNDING ONLY ACTION,https://www.fpds.gov/ezsearch/jsp/viewLinkCont...,2025-05-30,7.305833e+05,0


In [21]:
#identified the range of the deleted_date field
contracts_2['early_deletion'].describe()

,early_deletion
count,11042.000000
mean,0.239359
std,0.426711
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,1.000000


In [27]:
df_mod3.head()

,Unnamed: 0,award_id_piid,total_dollars_obligated,small_agricultural_cooperative,hospital_flag,for_profit_organization,foreign_owned,planning_commission,interstate_entity,manufacturer_of_goods,...,misc_minority,disadv_biz,LLCs_solep_scorp,any_race_group,hispanic,veteran,not_for_profits,authorities,corporate,ability
0,0,47QSWA20D0092,0.0,0,0,1,0,0,0,1,...,0,0,0,0,0,0,0,0,1,0
1,1,47QRAA24D006P,0.0,0,0,1,0,0,0,0,...,0,1,0,0,0,0,0,0,1,0
2,2,47QTCA20D008C,0.0,0,0,1,0,0,0,0,...,0,0,1,0,0,0,0,0,1,0
3,3,47QTCA23D0009,0.0,0,0,1,0,0,0,0,...,0,0,1,0,0,1,0,0,1,0
4,4,GS33F004DA,0.0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0


In [24]:
copy = df_mod3.copy()
columns_to_keep_indices_mod = [2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29]
copy = copy.iloc[:, columns_to_keep_indices_mod]
copy.head()

,total_dollars_obligated,small_agricultural_cooperative,hospital_flag,for_profit_organization,foreign_owned,planning_commission,interstate_entity,manufacturer_of_goods,international_organization,labor_surplus_area_firm,...,misc_minority,disadv_biz,LLCs_solep_scorp,any_race_group,hispanic,veteran,not_for_profits,authorities,corporate,ability
0,0.0,0,0,1,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,1,0
1,0.0,0,0,1,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,1,0
2,0.0,0,0,1,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,1,0
3,0.0,0,0,1,0,0,0,0,0,0,...,0,0,1,0,0,1,0,0,1,0
4,0.0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0


In [ ]:
copy_filtered = copy[copy['total_dollars_obligated'] != 0].copy()

In [ ]:
copy_filtered.head()

,total_dollars_obligated,small_agricultural_cooperative,hospital_flag,for_profit_organization,foreign_owned,planning_commission,interstate_entity,manufacturer_of_goods,international_organization,labor_surplus_area_firm,...,misc_minority,disadv_biz,LLCs_solep_scorp,any_race_group,hispanic,veteran,not_for_profits,authorities,corporate,ability
153,152077.74,0,0,1,0,0,0,0,0,0,...,1,1,1,1,0,0,0,0,1,0
169,6949.72,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,1,0,1,1
174,10697.33,0,0,1,0,0,0,0,0,0,...,1,1,1,1,0,0,0,0,1,0
217,9112069.09,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,1,0
277,30150.18,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,1,0


In [34]:
copy2 = df_mod3.copy()
columns_to_keep_indices_mod = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29]
copy2 = copy2.iloc[:, columns_to_keep_indices_mod]
copy2.head()
#join copy2 to contracts2 by piid field, keeping all of the fields in copy2, and keeping only the early_deletion field from contracts2. inner merge
copy2 = copy2.merge(contracts_2[['piid', 'early_deletion']], left_on='award_id_piid', right_on='piid', how='inner')
copy2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13602 entries, 0 to 13601
Data columns (total 31 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   award_id_piid                   13602 non-null  object 
 1   total_dollars_obligated         13602 non-null  float64
 2   small_agricultural_cooperative  13602 non-null  int64  
 3   hospital_flag                   13602 non-null  int64  
 4   for_profit_organization         13602 non-null  int64  
 5   foreign_owned                   13602 non-null  int64  
 6   planning_commission             13602 non-null  int64  
 7   interstate_entity               13602 non-null  int64  
 8   manufacturer_of_goods           13602 non-null  int64  
 9   international_organization      13602 non-null  int64  
 10  labor_surplus_area_firm         13602 non-null  int64  
 11  DOGE_Flag                       13602 non-null  int64  
 12  Red State                       

In [38]:
#remove all rows from copy2 where each value is 0
copy2 = copy2[(copy2 != 0).any(axis=1)]
copy2.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13602 entries, 0 to 13601
Data columns (total 31 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   award_id_piid                   13602 non-null  object 
 1   total_dollars_obligated         13602 non-null  float64
 2   small_agricultural_cooperative  13602 non-null  int64  
 3   hospital_flag                   13602 non-null  int64  
 4   for_profit_organization         13602 non-null  int64  
 5   foreign_owned                   13602 non-null  int64  
 6   planning_commission             13602 non-null  int64  
 7   interstate_entity               13602 non-null  int64  
 8   manufacturer_of_goods           13602 non-null  int64  
 9   international_organization      13602 non-null  int64  
 10  labor_surplus_area_firm         13602 non-null  int64  
 11  DOGE_Flag                       13602 non-null  int64  
 12  Red State                       

In [39]:
#save copy2 as a csv to google drive called df_mod4
copy2.to_csv('/content/drive/MyDrive/DOGE/df_mod4.csv', index=False)

In [36]:
copy2.head()

,award_id_piid,total_dollars_obligated,small_agricultural_cooperative,hospital_flag,for_profit_organization,foreign_owned,planning_commission,interstate_entity,manufacturer_of_goods,international_organization,...,LLCs_solep_scorp,any_race_group,hispanic,veteran,not_for_profits,authorities,corporate,ability,piid,early_deletion
0,75D30124F20209,3651042.26,0,0,1,0,0,0,0,0,...,0,0,0,1,0,0,1,0,75D30124F20209,0
1,75D30124P18927,57673.50,0,0,1,0,0,0,0,0,...,1,0,0,0,0,0,1,0,75D30124P18927,0
2,75N95023F00001,748487.00,0,0,1,0,0,0,0,0,...,1,0,0,0,0,0,1,0,75N95023F00001,0
3,75N95023F00001,748487.00,0,0,1,0,0,0,0,0,...,1,0,0,0,0,0,1,0,75N95023F00001,0
4,75N95023F00001,748487.00,0,0,1,0,0,0,0,0,...,1,0,0,0,0,0,1,0,75N95023F00001,0


In [42]:
copy2.drop('total_dollars_obligated', axis=1, inplace=True)
copy2.drop('award_id_piid', axis=1, inplace=True)
copy2.drop('misc_minority', axis=1, inplace=True)
copy2.drop('piid', axis=1, inplace=True)
copy2.drop('DOGE_Flag', axis=1, inplace=True)
copy2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13602 entries, 0 to 13601
Data columns (total 28 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   small_agricultural_cooperative  13602 non-null  int64 
 1   hospital_flag                   13602 non-null  int64 
 2   for_profit_organization         13602 non-null  int64 
 3   foreign_owned                   13602 non-null  int64 
 4   planning_commission             13602 non-null  int64 
 5   interstate_entity               13602 non-null  int64 
 6   manufacturer_of_goods           13602 non-null  int64 
 7   international_organization      13602 non-null  int64 
 8   labor_surplus_area_firm         13602 non-null  int64 
 9   DOGE_Flag                       13602 non-null  int64 
 10  Red State                       13602 non-null  int64 
 11  higher_ed                       13602 non-null  int64 
 12  native                          13602 non-null

In [45]:
copy2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13602 entries, 0 to 13601
Data columns (total 26 columns):
 #   Column                          Non-Null Count  Dtype
---  ------                          --------------  -----
 0   small_agricultural_cooperative  13602 non-null  int64
 1   hospital_flag                   13602 non-null  int64
 2   for_profit_organization         13602 non-null  int64
 3   foreign_owned                   13602 non-null  int64
 4   planning_commission             13602 non-null  int64
 5   interstate_entity               13602 non-null  int64
 6   manufacturer_of_goods           13602 non-null  int64
 7   international_organization      13602 non-null  int64
 8   labor_surplus_area_firm         13602 non-null  int64
 9   Red State                       13602 non-null  int64
 10  higher_ed                       13602 non-null  int64
 11  native                          13602 non-null  int64
 12  API                             13602 non-null  int64
 13  W

In [47]:
#linear regression
X = copy2.drop('early_deletion', axis = 1)
y = copy2['early_deletion']

In [48]:
def rmspe(y_test, y_pred):
  '''
  This function takes y_test and y_pred and calculates the RMSPE'''
  return np.sqrt(np.mean(np.square((y_test - y_pred) / y_test)))

In [49]:
# Perform CV
n = 500
results = np.zeros(n)
for idx in range(n):
  X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.25)
  model = LinearRegression()
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)
  results[idx] = rmspe(y_test,y_pred)
print(f"CV RMSPE: {results.mean().round(2)}%")
print(f"Number of Predictors: {len(X.columns)}")

CV RMSPE: inf%
Number of Predictors: 25


In [52]:
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer

# Define RMSPE function
def rmspe(y_true, y_pred):
    """Calculate Root Mean Square Percentage Error."""
    percentage_errors = (y_true - y_pred) / y_true
    return np.sqrt(np.mean(percentage_errors**2))

# Create a custom scorer for cross-validation
rmspe_scorer = make_scorer(rmspe, greater_is_better=False)

# Example: Using cross-validation with a regression model
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from sklearn.datasets import make_regression

# Generate synthetic regression data
X, y = make_regression(n_samples=100, n_features=5, noise=0.1, random_state=42)

# Initialize a regression model
model = LinearRegression()

# Perform cross-validation
cv = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, scoring=rmspe_scorer, cv=cv)

# Output RMSPE scores
print("RMSPE scores for each fold:", -scores)  # Negate because `greater_is_better=False`
print("Mean RMSPE:", -np.mean(scores))


# Calculate the correlation matrix
correlation_matrix = copy.corr()
target_correlation = correlation_matrix['total_dollars_obligated'].drop('total_dollars_obligated')
print("Correlation Coefficients with 'total_dollars_obligated':")
print(target_correlation)
# Calculate the correlation matrix
correlation_matrix = copy.corr()

# Get the correlations with the target variable 'total_dollars_obligated'
target_correlation = correlation_matrix['total_dollars_obligated'].drop('total_dollars_obligated')

# Display the correlations
print("Correlation Coefficients with 'total_dollars_obligated':")
print(target_correlation)

RMSPE scores for each fold: [0.00801599 0.00927796 0.00166041 0.00234072 0.91331708]
Mean RMSPE: 0.18692243170845985
Correlation Coefficients with 'total_dollars_obligated':
small_agricultural_cooperative   -0.000405
hospital_flag                    -0.000838
for_profit_organization          -0.026174
foreign_owned                    -0.002983
planning_commission              -0.000290
interstate_entity                -0.000168
manufacturer_of_goods            -0.008951
international_organization       -0.001067
labor_surplus_area_firm          -0.000280
DOGE_Flag                        -0.001736
Red State                         0.003713
higher_ed                         0.060167
native                           -0.004393
API                              -0.005056
Women                            -0.008670
governments                       0.006206
black                            -0.003876
misc_minority                    -0.003683
disadv_biz                       -0.010414
LLCs_sole

In [53]:
# Fit a linear model using statsmodels
def fit_linear_model(endog, exog):
  if endog.dtype == object or exog is not None and exog.dtype == object:
      raise ValueError("Pandas data cast to numpy dtype of object. "
                      "Check input data with np.asarray(data).")
  else:
      myfit = sm.OLS(y_train, X_train).fit()
      myfit.summary()


In [54]:
myfit = sm.OLS(y_train, X_train).fit()
myfit.summary()

/usr/local/lib/python3.11/dist-packages/statsmodels/regression/linear_model.py:1966: RuntimeWarning: divide by zero encountered in scalar divide
  return np.sqrt(eigvals[0]/eigvals[-1])


<class 'statsmodels.iolib.summary.Summary'>
"""
                                 OLS Regression Results                                
=======================================================================================
Dep. Variable:         early_deletion   R-squared (uncentered):                   0.230
Model:                            OLS   Adj. R-squared (uncentered):              0.228
Method:                 Least Squares   F-statistic:                              144.7
Date:                Wed, 25 Jun 2025   Prob (F-statistic):                        0.00
Time:                        19:47:49   Log-Likelihood:                         -5285.3
No. Observations:               10201   AIC:                                  1.061e+04
Df Residuals:                   10180   BIC:                                  1.076e+04
Df Model:                          21                                                  
Covariance Type:            nonrobust                                                  
==================================================================================================
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
small_agricultural_cooperative          0          0        nan        nan           0           0
hospital_flag                     -0.0051      0.047     -0.107      0.915      -0.098       0.088
for_profit_organization            0.2343      0.014     17.136      0.000       0.207       0.261
foreign_owned                      0.0475      0.021      2.216      0.027       0.005       0.090
planning_commission            -9.872e-17   3.72e-17     -2.653      0.008   -1.72e-16   -2.58e-17
interstate_entity              -2.786e-16   1.11e-16     -2.521      0.012   -4.95e-16   -6.19e-17
manufacturer_of_goods             -0.0811      0.022     -3.664      0.000      -0.125      -0.038
international_organization        -0.0224      0.079     -0.283      0.777      -0.178       0.133
labor_surplus_area_firm         4.291e-17    6.9e-17      0.622      0.534   -9.24e-17    1.78e-16
Red State                          0.0103      0.008      1.225      0.221      -0.006       0.027
higher_ed                         -0.1192      0.025     -4.704      0.000      -0.169      -0.070
native                            -0.1499      0.059     -2.524      0.012      -0.266      -0.033
API                               -0.1741      0.058     -2.988      0.003      -0.288      -0.060
Women                             -0.0086      0.011     -0.773      0.439      -0.031       0.013
governments                        0.1664      0.030      5.524      0.000       0.107       0.225
black                             -0.1187      0.058     -2.028      0.043      -0.233      -0.004
disadv_biz                         0.0113      0.015      0.777      0.437      -0.017       0.040
LLCs_solep_scorp                   0.0510      0.011      4.554      0.000       0.029       0.073
any_race_group                     0.1098      0.058      1.884      0.060      -0.004       0.224
hispanic                          -0.1071      0.060     -1.772      0.076      -0.226       0.011
veteran                           -0.1051      0.014     -7.568      0.000      -0.132      -0.078
not_for_profits                    0.2425      0.017     14.551      0.000       0.210       0.275
authorities                        0.8336      0.237      3.522      0.000       0.370       1.298
corporate                         -0.0375      0.011     -3.489      0.000      -0.059      -0.016
ability                           -0.1731      0.155     -1.121      0.262      -0.476       0.130
==============================================================================
Omnibus:                     1835.355   Durbin-Watson:                   1.980
Prob(Omnibus):                  0.000   Jarque-Bera (J

In [56]:
#exporting the results from above into a a df with coef, std err, p, .025, and .975 as a table
results_df
results_df = pd.DataFrame({
    'coef': myfit.params,
    'std err': myfit.bse,
    'p': myfit.pvalues,
    '0.025': myfit.conf_int()[0],
    '0.975': myfit.conf_int()[1]
})
results_df

,coef,std err,p,0.025,0.975
small_agricultural_cooperative,0.000000e+00,0.000000e+00,NaN,0.000000e+00,0.000000e+00
hospital_flag,-5.058400e-03,4.724883e-02,9.147445e-01,-9.767542e-02,8.755862e-02
for_profit_organization,2.342588e-01,1.367028e-02,6.438423e-65,2.074623e-01,2.610552e-01
foreign_owned,4.754893e-02,2.145556e-02,2.670263e-02,5.491810e-03,8.960605e-02
planning_commission,-9.871721e-17,3.721407e-17,7.997819e-03,-1.716641e-16,-2.577030e-17
interstate_entity,-2.785516e-16,1.105096e-16,1.173031e-02,-4.951722e-16,-6.193103e-17
manufacturer_of_goods,-8.111716e-02,2.214091e-02,2.498811e-04,-1.245177e-01,-3.771662e-02
international_organization,-2.241505e-02,7.928373e-02,7.773977e-01,-1.778268e-01,1.329967e-01
labor_surplus_area_firm,4.291269e-17,6.901767e-17,5.341111e-01,-9.237555e-17,1.782009e-16
Red State,1.034685e-02,8.446632e-03,2.206157e-01,-6.210212e-03,2.690391e-02
